In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# Full tables (optional peek); pipeline reloads in the next cell.
news_labels_full = pd.read_csv("news_topics.csv")
news_train_full = pd.read_csv("news.csv")
news_labels_full.shape, news_train_full.shape

((2606875, 3), (23149, 2))

In [3]:
news_test = pd.read_csv("news_test.csv")

print(news_test)

            id                                            article
0        26152   world world world world qualif qualif sunday ...
1        26154   beat beat beat world world world world world ...
2        26156   open nick knight knight knight hit hit hundr ...
3        26158   world sunday minut put athlet komen record da...
4        26160   world sunday minut athlet komen komen break r...
...        ...                                                ...
390611  810926   ist citibank centur rate contribut contribut ...
390612  810928   open ist citibank centur rate contribut contr...
390613  810930   open heavy heavy colomb phon gmt figur helico...
390614  810932   load parcel parcel cent declin weekend ups up...
390615  810934   hit deutschemark entry entry complet rate rat...

[390616 rows x 2 columns]


In [4]:
N = 5000
news_train = pd.read_csv("news.csv")
news_labels = pd.read_csv("news_topics.csv")
news_labels["cat"] = news_labels["cat"].str.strip()

articles_5k = news_train.head(N)
ids_5k = set(articles_5k["id"])

# TF-IDF — text is already stemmed / cleaned, just split on whitespace
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(articles_5k["article"])
print(f"X_tfidf: {X_tfidf.shape}")

# One-hot labels: one row per article, columns = all unique categories
labels_sub = news_labels[news_labels["id"].isin(ids_5k)]
cat_lists = labels_sub.groupby("id", sort=False)["cat"].apply(list)
labels_5k = articles_5k[["id"]].merge(cat_lists.rename("cats"), on="id", how="left")
labels_5k["cats"] = labels_5k["cats"].apply(lambda x: x if isinstance(x, list) else [])

mlb = MultiLabelBinarizer()
Y_labels = mlb.fit_transform(labels_5k["cats"])
print(f"Y_labels: {Y_labels.shape}  ({len(mlb.classes_)} unique categories)")

Y_labels

X_tfidf: (5000, 21871)
Y_labels: (5000, 95)  (95 unique categories)


array([[0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [5]:
# (predictions moved to base_level.ipynb)

Y: (5000, 4)  counts={'CCAT': 2471, 'ECAT': 826, 'GCAT': 1291, 'MCAT': 1277}
PCA: (5000, 21871) → (5000, 300), var=0.438
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done

=== SVM Baseline — H1 Topics (5-fold CV) ===

Topic  Accuracy  Precision   Recall       F1
 CCAT    0.9208   0.918423 0.921678 0.919879
 ECAT    0.9304   0.862994 0.689690 0.765999
 GCAT    0.9510   0.939832 0.864908 0.900703
 MCAT    0.9532   0.936523 0.876437 0.905425

Macro-avg F1: 0.8730

=== Confusion Matrices (summed over all folds) ===

CCAT:
          Pred 0  Pred 1
Actual 0    2326     203
Actual 1     193    2278

ECAT:
          Pred 0  Pred 1
Actual 0    4082      92
Actual 1     256     570

GCAT:
          Pred 0  Pred 1
Actual 0    3638      71
Actual 1     174    1117

MCAT:
          Pred 0  Pred 1
Actual 0    3647      76
Actual 1     158    1119


In [6]:
# (predictions moved to base_level.ipynb)

Test articles: 390616


KeyboardInterrupt: 